In [ ]:
# ============================================
# Memory-efficient MintFlow perturbation script
# Runs perturbation only on selected sections
# instead of concatenating all 880k cells
# Includes:
#   1) generation without guidance
#   2) generation with gene guidance
#   3) diagnostic plots
#   4) per-section output saving
# ============================================

import os
import gc
import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import squidpy as sq
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import mintflow

# --------------------------------------------
# USER SETTINGS
# --------------------------------------------
path_results = "./perturbation_results_gene_programm/"
os.makedirs(path_results, exist_ok=True)

path_dump_root = "./NonGit/Outputs_GenePerrurbation_geneprogramm"
os.makedirs(path_dump_root, exist_ok=True)

obskey_celltype = "lvl5_annotation"

# Reference batch index for size factor estimation.
# Fixed to 17 = BK25_Lesional Baseline, used for all sections.
REFERENCE_BATCH_INDEX = 17

num_iters_guidance = 1000
lr_optim = 0.0001
conf_interval_percelltype_prior = 0.95
num_DEGs_to_show = 10
num_randselect_elements_to_inspect = 10

list_genes_KO = ["IL4R","IL13RA1","CCL26","ALOX15","CCL17","CCL13","CCL18","CCL27","POSTN"]

fname_mintflow_checkpoint = (
    "/nfs/team361/am74/mintflow/revision/skin_perturbation/"
    "mintflow_runs/run1_tiger/model_epoch_7.pt"
)

fname_data_mintflow = "./mintflow_runs/run1_tiger/run1data_mintflow.pkl"

path_anndata = (
    "/nfs/team361/am74/mintflow/revision/skin_perturbation/per_section_raw/"
)

# Sections to perturb
selected_sample_ids = {
    "BK18_Lesional Baseline",
    "BK23_Lesional Baseline",
    "BK24_Lesional Baseline",
    "BK25_Lesional Baseline",
    "BK27_Lesional Baseline",
    "BK30_Lesional Baseline",
    "BK39_Lesional Baseline",
    "BK20_Lesional Baseline",
    "BK21_Lesional Baseline",
    "BK22_Lesional Baseline",
}

# Known batch index map (from data_mintflow, printed at startup)
# batch_index_trainingdata is per-section (its own embedding)
# estimate_spatial_sizefactors_on_sections is always REFERENCE_BATCH_INDEX
KNOWN_BATCH_MAP = {
    "BK18_Lesional Baseline": 0,
    "BK20_Lesional Baseline": 3,
    "BK21_Lesional Baseline": 6,
    "BK22_Lesional Baseline": 9,
    "BK23_Lesional Baseline": 11,
    "BK24_Lesional Baseline": 14,
    "BK25_Lesional Baseline": 17,
    "BK27_Lesional Baseline": 20,
    "BK30_Lesional Baseline": 24,
    "BK39_Lesional Baseline": 27,
}

# spatial graph settings for generation adata
kwargs_neighbourhood_graph = {
    "spatial_key": "spatial",
    "library_key": None,
    "set_diag": False,
    "delaunay": False,
    "n_neighs": 5,
}


# --------------------------------------------
# HELPERS
# --------------------------------------------

def sanitize_name(s):
    return str(s).replace(" ", "_").replace("/", "_")


def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def find_matching_files_by_sample_id(folder_path, selected_ids):
    folder = Path(folder_path)
    files = sorted(folder.glob("*.h5ad"))
    matched = []

    print(f"Scanning {len(files)} .h5ad files for selected sample_id values...")

    for f in tqdm(files, desc="Scanning files"):
        try:
            a = sc.read_h5ad(f)
            if "sample_id" not in a.obs.columns:
                del a
                gc.collect()
                continue

            present_ids = set(a.obs["sample_id"].astype(str).unique())
            overlap = present_ids.intersection(selected_ids)
            if len(overlap) > 0:
                matched.append((f, sorted(list(overlap))))

            del a
            gc.collect()

        except Exception as e:
            print(f"[WARN] Could not inspect file {f}: {e}")

    return matched


def load_selected_cells_from_file(file_path, selected_ids):
    a = sc.read_h5ad(file_path)

    if "sample_id" not in a.obs.columns:
        raise KeyError(f"'sample_id' not found in obs for {file_path}")

    mask = a.obs["sample_id"].astype(str).isin(selected_ids).to_numpy()
    if mask.sum() == 0:
        del a
        gc.collect()
        return None

    a = a[mask].copy()
    return a


def ensure_spatial_obsm(adata):
    if "spatial" in adata.obsm:
        return adata

    if "x_centroid" in adata.obs.columns and "y_centroid" in adata.obs.columns:
        adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].to_numpy()
        return adata

    raise KeyError(
        "No adata.obsm['spatial'] found, and could not create it because "
        "'x_centroid'/'y_centroid' are missing from .obs."
    )


def add_mintflow_required_obs_columns(adata):
    if "sample_id__xenium_id_recoded" not in adata.obs.columns:
        raise KeyError(
            "'sample_id__xenium_id_recoded' not found in obs; "
            "needed for MintFlow config fields in your original script."
        )

    adata.obs["TissueSectionID_for_MintFlow"] = (
        adata.obs["sample_id__xenium_id_recoded"].astype("category")
    )
    adata.obs["batchID_for_MintFlow"] = (
        adata.obs["sample_id__xenium_id_recoded"].astype("category")
    )
    return adata


def build_gene_perturbation_df(adata, ko_genes, target_mask):
    n_cells, n_genes = adata.shape

    mat = np.full((n_cells, n_genes), "DC", dtype="U2")

    gene_idx = [
        adata.var_names.get_loc(g)
        for g in ko_genes
        if g in adata.var_names
    ]

    missing = [g for g in ko_genes if g not in adata.var_names]
    if len(missing) > 0:
        print(f"[WARN] These KO genes are missing in this section and will be skipped: {missing}")

    if len(gene_idx) == 0:
        raise ValueError("None of the requested KO genes were found in adata.var_names.")

    mat[np.ix_(target_mask, gene_idx)] = "KO"

    df_gene_perturbation = pd.DataFrame(
        mat,
        index=adata.obs_names,
        columns=adata.var_names
    )
    return df_gene_perturbation


def build_configs_for_all_files(path_anndata):
    config_data_train, config_data_evaluation, config_model, config_training = (
        mintflow.get_default_configurations(
            num_tissue_sections_training=64,
            num_tissue_sections_evaluation=64,
        )
    )

    files = sorted([f for f in os.listdir(path_anndata) if f.endswith(".h5ad")])

    for idx, filename in enumerate(files, start=1):
        key = f"anndata{idx}"
        filepath = os.path.join(path_anndata, filename)

        for cfg in [config_data_train, config_data_evaluation]:
            cfg["list_tissue"][key]["file"] = filepath
            cfg["list_tissue"][key]["obskey_cell_type"] = "lvl5_annotation"
            cfg["list_tissue"][key]["obskey_sliceid_to_checkUnique"] = "sample_id__xenium_id_recoded"
            cfg["list_tissue"][key]["obskey_x"] = "x_centroid"
            cfg["list_tissue"][key]["obskey_y"] = "y_centroid"
            cfg["list_tissue"][key]["obskey_biological_batch_key"] = "sample_id__xenium_id_recoded"
            cfg["list_tissue"][key]["config_neighbourhood_graph"] = {
                "n_neighs": 10,
                "set_diag": "False",
                "delaunay": "False",
            }

        config_data_train["list_tissue"][key]["config_dataloader_train"]["width_window"] = 1368
        config_data_evaluation["list_tissue"][key]["config_dataloader_test"]["width_window"] = 1368

    config_model["coef_xbarint2notbatchID_loss"] = 1.0
    config_model["coef_xbarspl2notbatchID_loss"] = 1.0
    config_model["coef_flowmatchingloss"] = 0.0
    config_model["dict_qname_to_scaleandunweighted"] = (
        "impanddisentgl_int#0.1#True&"
        "impanddisentgl_spl#0.0#True&"
        "varphi_enc_int#0.0#True&"
        "varphi_enc_spl#0.0#True&"
        "z#0.0#True&"
        "sin#0.0#True&"
        "sout#0.0#True"
    )
    config_model["coef_loss_CTpredfromZ"] = 100

    config_training["num_training_epochs"] = 8
    config_training["flag_use_GPU"] = "True"
    config_training["flag_enable_wandb"] = "True"
    config_training["annealing_decoder_XintXspl_coef_max"] = 0.01
    config_training["wandb_project_name"] = "MintFlow_skin_AD"
    config_training["wandb_run_name"] = "Mintflow_skin_run1"

    config_data_train = mintflow.verify_and_postprocess_config_data_train(config_data_train)
    config_data_evaluation = mintflow.verify_and_postprocess_config_data_evaluation(config_data_evaluation)
    config_model = mintflow.verify_and_postprocess_config_model(
        config_model,
        num_tissue_sections=len(config_data_train),
    )
    config_training = mintflow.verify_and_postprocess_config_training(config_training)

    return {
        "config_data_train": config_data_train,
        "config_data_evaluation": config_data_evaluation,
        "config_model": config_model,
        "config_training": config_training,
    }


def get_batch_index_for_section(sid, data_mintflow):
    """
    Look up the integer batch index for a given sample_id (sid).
    First tries the hardcoded KNOWN_BATCH_MAP (verified from printed map),
    then falls back to dynamic lookup from data_mintflow using
    startswith(sid + '__') to match the full xenium key format.
    Returns REFERENCE_BATCH_INDEX as last resort.
    """
    # 1. Try hardcoded map first (fastest, most reliable)
    if sid in KNOWN_BATCH_MAP:
        idx = KNOWN_BATCH_MAP[sid]
        print(f"[INFO] '{sid}' -> batch index {idx} (from KNOWN_BATCH_MAP)")
        return idx

    # 2. Dynamic lookup from data_mintflow
    try:
        batch_map = data_mintflow["train_list_tissue_section"].map_Batchname_to_inflowBatchID
        for key, idx in batch_map.items():
            if key.startswith(sid + "__"):
                print(f"[INFO] '{sid}' -> training key '{key}' -> batch index {idx} (dynamic lookup)")
                return idx
        print(f"[WARN] Could not find '{sid}' in batch map.")
    except Exception as e:
        print(f"[WARN] Could not retrieve batch map: {e}")

    # 3. Fall back to reference index
    print(f"[WARN] Falling back to REFERENCE_BATCH_INDEX={REFERENCE_BATCH_INDEX} (BK25_Lesional Baseline)")
    return REFERENCE_BATCH_INDEX


def maybe_move_model_to_device(model, device):
    if hasattr(model, "to"):
        model = model.to(device)
    if hasattr(model, "eval"):
        model.eval()
    return model


def save_loss_curve(list_overall_loss, out_dir):
    plt.figure()
    plt.plot(range(len(list_overall_loss)), list_overall_loss)
    plt.xlabel("iteration index of updating embeddings")
    plt.ylabel("overall loss")
    plt.savefig(
        os.path.join(out_dir, "history_overall_loss.png"),
        bbox_inches="tight",
        pad_inches=0,
    )
    plt.close()


def save_projection_plots(list_trackinginfo_projection, out_dir):
    for str_embedding in ["z", "sout"]:
        key = f"{str_embedding}_flag_do_project"
        try:
            np_frac_projection_happened = np.stack(
                [u[key] for u in list_trackinginfo_projection]
            ).mean(1)
        except Exception as e:
            print(f"[WARN] Could not compute projection plot for {str_embedding}: {e}")
            continue

        plt.figure()
        plt.plot(range(np_frac_projection_happened.shape[0]), np_frac_projection_happened)
        plt.xlabel(
            f"iteration index of updating embedding "
            f"{'Z' if str_embedding == 'z' else 'Sout'}"
        )
        plt.ylabel("fraction of cells projected to confidence interval")
        plt.title(f"embedding: {'Z' if str_embedding == 'z' else 'Sout'}")
        plt.savefig(
            os.path.join(out_dir, f"fraction_projections_{str_embedding}.png"),
            bbox_inches="tight",
            pad_inches=0,
        )
        plt.close()


def save_sample_loss_traces(
    list_trackinginfo_computeloss,
    df_gene_perturbation,
    out_dir,
    num_randselect_elements_to_inspect=10,
):
    if len(list_trackinginfo_computeloss) == 0:
        print("[WARN] list_trackinginfo_computeloss is empty; skipping element-wise loss plots.")
        return

    keys0 = list(list_trackinginfo_computeloss[0].keys())
    valid_keys, list_cellindex, list_geneindex = [], [], []

    for k in keys0:
        if not (k.startswith("cellindex_") and "_geneindex_" in k):
            continue
        try:
            parts = k.split("_")
            cidx = int(parts[1])
            gidx = int(parts[3])
            valid_keys.append(k)
            list_cellindex.append(cidx)
            list_geneindex.append(gidx)
        except Exception:
            continue

    if len(valid_keys) == 0:
        print("[WARN] No per-cell-gene loss keys found; skipping element-wise loss plots.")
        return

    non_dc_indices = []
    for i, (cidx, gidx) in enumerate(zip(list_cellindex, list_geneindex)):
        try:
            if df_gene_perturbation.iloc[cidx, gidx] != "DC":
                non_dc_indices.append(i)
        except Exception:
            pass

    if len(non_dc_indices) == 0:
        print("[WARN] No non-DC entries found among tracked loss elements; skipping element-wise loss plots.")
        return

    choose_n = min(num_randselect_elements_to_inspect, len(non_dc_indices))
    idx_cellgene_randselect = sorted(
        np.random.choice(non_dc_indices, size=choose_n, replace=False).tolist()
    )

    for idx_cellgene in idx_cellgene_randselect:
        cidx = list_cellindex[idx_cellgene]
        gidx = list_geneindex[idx_cellgene]
        key = f"cellindex_{cidx}_geneindex_{gidx}"

        plt.figure()
        plt.plot(
            range(len(list_trackinginfo_computeloss)),
            [t[key] for t in list_trackinginfo_computeloss]
        )
        plt.xlabel("iteration index of updating embeddings")
        plt.ylabel("loss for the specific cell and gene")
        plt.title(f"cell:{cidx}, gene:{gidx}, pert:{df_gene_perturbation.iloc[cidx, gidx]}")
        plt.savefig(
            os.path.join(out_dir, f"sample_history_loss_elem_{cidx}_{gidx}.png"),
            bbox_inches="tight",
            pad_inches=0,
        )
        plt.close()


# --------------------------------------------
# LOAD MODEL + DATA ONCE
# --------------------------------------------
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

sys.modules["inflow"] = mintflow

print("Loading model checkpoint...")
model = torch.load(
    fname_mintflow_checkpoint,
    weights_only=False,
    map_location=device,
)
model = maybe_move_model_to_device(model, device)

print("Loading data_mintflow...")
with open(fname_data_mintflow, "rb") as f:
    data_mintflow = pickle.load(f)

# ---- Print the full batch map immediately after loading ----
print("\nBatch index map from training data:")
try:
    batch_map = data_mintflow["train_list_tissue_section"].map_Batchname_to_inflowBatchID
    for name, idx in sorted(batch_map.items(), key=lambda x: x[1]):
        print(f"  {idx:>3}  {name}")
except Exception as e:
    print(f"[WARN] Could not print batch map: {e}")

print(f"\nReference batch index for size factor estimation: {REFERENCE_BATCH_INDEX} (BK25_Lesional Baseline)")

print("\nBuilding MintFlow configs...")
dict_all4_configs = build_configs_for_all_files(path_anndata)

# --------------------------------------------
# FIND ONLY RELEVANT FILES
# --------------------------------------------
matched_files = find_matching_files_by_sample_id(path_anndata, selected_sample_ids)

if len(matched_files) == 0:
    raise RuntimeError("No .h5ad files found containing the requested sample_id values.")

print("\nMatched files:")
for f, overlap in matched_files:
    print(" -", f.name, "->", overlap)

# --------------------------------------------
# RUN SECTION-BY-SECTION
# --------------------------------------------
all_results = {}

for file_path, overlap_ids in matched_files:
    print("\n" + "=" * 80)
    print(f"Processing file: {file_path.name}")
    print(f"Matching sample_id values in this file: {overlap_ids}")

    adata_file = load_selected_cells_from_file(file_path, selected_sample_ids)
    if adata_file is None:
        print("No selected cells found after filtering; skipping.")
        continue

    sample_ids_in_file = sorted(
        set(adata_file.obs["sample_id"].astype(str).unique()).intersection(selected_sample_ids)
    )

    for sid in sample_ids_in_file:
        print("\n" + "-" * 80)
        print(f"Processing section: {sid}")

        # batch_index_trainingdata: per-section own embedding index
        # estimate_spatial_sizefactors_on_sections: always fixed to REFERENCE_BATCH_INDEX
        batch_index_trainingdata = get_batch_index_for_section(sid, data_mintflow)
        print(f"  batch_index_trainingdata : {batch_index_trainingdata} (own embedding)")
        print(f"  size factor reference    : {REFERENCE_BATCH_INDEX} (BK25_Lesional Baseline)")

        out_prefix = sanitize_name(sid)
        out_dir = os.path.join(path_results, out_prefix)
        os.makedirs(out_dir, exist_ok=True)

        try:
            adata_sec = adata_file[adata_file.obs["sample_id"].astype(str) == sid].copy()

            if adata_sec.n_obs == 0:
                print("No cells in this section after filtering; skipping.")
                del adata_sec
                gc.collect()
                continue

            print(f"Section shape: cells={adata_sec.n_obs}, genes={adata_sec.n_vars}")

            adata_sec = add_mintflow_required_obs_columns(adata_sec)
            adata_sec = ensure_spatial_obsm(adata_sec)

            adata_sec.uns = {}
            sq.gr.spatial_neighbors(adata=adata_sec, **kwargs_neighbourhood_graph)

            target_mask = np.ones(adata_sec.n_obs, dtype=bool)

            df_gene_perturbation = build_gene_perturbation_df(
                adata=adata_sec,
                ko_genes=list_genes_KO,
                target_mask=target_mask,
            )

            pd.DataFrame({
                "obs_name": adata_sec.obs_names,
                "sample_id": adata_sec.obs["sample_id"].astype(str).values,
            }).to_csv(os.path.join(out_dir, "cell_metadata.csv"), index=False)

            print("Running MintFlow generation without guidance...")
            with torch.no_grad():
                result_generation_without_guidance = mintflow.generate_insilico_ST_data(
                    adata=adata_sec,
                    obskey_celltype=obskey_celltype,
                    obspkey_neighbourhood_graph="spatial_connectivities",
                    device=device,
                    batch_index_trainingdata=batch_index_trainingdata,
                    num_generated_realisations=3,
                    model=model,
                    data_mintflow=data_mintflow,
                    dict_all4_configs=dict_all4_configs,
                    estimate_spatial_sizefactors_on_sections=[REFERENCE_BATCH_INDEX],
                )

            save_pickle(
                result_generation_without_guidance,
                os.path.join(out_dir, "result_generation_without_guidance.pkl")
            )

            adata_insilico_gen_wihout_guidance = adata_sec.copy()
            adata_insilico_gen_wihout_guidance.X = (
                np.stack(
                    [d["MintFlow_Generated_Xint"] for d in result_generation_without_guidance["list_generated_realisations_ie_expressions"]],
                    0,
                ).mean(0)
                +
                np.stack(
                    [d["MintFLow_Generated_Xmic"] for d in result_generation_without_guidance["list_generated_realisations_ie_expressions"]],
                    0,
                ).mean(0)
            )

            adata_insilico_gen_wihout_guidance.obsm = {}
            for obsm_key in result_generation_without_guidance["list_generated_realisations_ie_expressions"][0].keys():
                adata_insilico_gen_wihout_guidance.obsm[obsm_key] = (
                    result_generation_without_guidance["list_generated_realisations_ie_expressions"][0][obsm_key]
                )

            del result_generation_without_guidance
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            print("Running MintFlow generation with gene perturbation guidance...")
            generated_realisation, adata_reference_expression, \
            list_overall_loss, list_trackinginfo_computeloss, \
            list_trackinginfo_projection, dict_debug_info = \
                mintflow.generate_insilico_ST_data_with_gene_perturbation(
                    adata_reference_expression=None,
                    df_gene_perturbation=df_gene_perturbation,
                    flag_doublecheck_df=True,
                    conf_interval_percelltype_prior=conf_interval_percelltype_prior,
                    dict_config_guidance_optimisation={
                        "type_optim": torch.optim.Adam,
                        "lr_optim": lr_optim,
                        "num_iters_guidance": num_iters_guidance,
                    },
                    adata=adata_sec,
                    obskey_celltype=obskey_celltype,
                    obspkey_neighbourhood_graph="spatial_connectivities",
                    device=device,
                    batch_index_trainingdata=batch_index_trainingdata,
                    model=model,
                    data_mintflow=data_mintflow,
                    dict_all4_configs=dict_all4_configs,
                    estimate_spatial_sizefactors_on_sections=[REFERENCE_BATCH_INDEX],
                )

            save_pickle(
                {
                    "generated_realisation": generated_realisation,
                    "adata_reference_expression": adata_reference_expression,
                    "list_overall_loss": list_overall_loss,
                    "list_trackinginfo_computeloss": list_trackinginfo_computeloss,
                    "list_trackinginfo_projection": list_trackinginfo_projection,
                    "dict_debug_info": dict_debug_info,
                },
                os.path.join(out_dir, "result_generation_with_guidance.pkl")
            )

            adata_insilico_gen_wih_guidance = adata_sec.copy()
            adata_insilico_gen_wih_guidance.X = (
                generated_realisation["MintFlow_Generated_Xint"] +
                generated_realisation["MintFLow_Generated_Xmic"]
            )
            adata_insilico_gen_wih_guidance.obsm = {}
            for k, v in generated_realisation.items():
                adata_insilico_gen_wih_guidance.obsm[k] = v

            save_loss_curve(list_overall_loss, out_dir)
            save_projection_plots(list_trackinginfo_projection, out_dir)
            save_sample_loss_traces(
                list_trackinginfo_computeloss=list_trackinginfo_computeloss,
                df_gene_perturbation=df_gene_perturbation,
                out_dir=out_dir,
                num_randselect_elements_to_inspect=num_randselect_elements_to_inspect,
            )

            adata_insilico_gen_wihout_guidance.obs["gene guidance"] = "gene guidance: False"
            adata_insilico_gen_wih_guidance.obs["gene guidance"] = "gene guidance: True"

            adata_with_and_without_guidance = ad.concat(
                [adata_insilico_gen_wihout_guidance, adata_insilico_gen_wih_guidance]
            )

            adata_insilico_gen_wihout_guidance.write_h5ad(
                os.path.join(out_dir, "adata_insilico_gen_without_guidance.h5ad")
            )
            adata_insilico_gen_wih_guidance.write_h5ad(
                os.path.join(out_dir, "adata_insilico_gen_with_guidance.h5ad")
            )
            adata_with_and_without_guidance.write_h5ad(
                os.path.join(out_dir, "adata_with_and_without_guidance.h5ad")
            )

            all_results[sid] = {
                "out_dir": out_dir,
                "batch_index_trainingdata": batch_index_trainingdata,
                "size_factor_reference_index": REFERENCE_BATCH_INDEX,
                "n_cells": int(adata_sec.n_obs),
                "n_genes": int(adata_sec.n_vars),
                "status": "success",
            }

            print(f"[OK] Finished section: {sid}")

        except RuntimeError as e:
            print(f"[ERROR] Runtime error in section {sid}: {e}")
            all_results[sid] = {"out_dir": out_dir, "status": "runtime_error", "error": str(e)}
            raise

        except Exception as e:
            print(f"[ERROR] Unexpected error in section {sid}: {e}")
            all_results[sid] = {"out_dir": out_dir, "status": "error", "error": str(e)}
            raise

        finally:
            for var_name in [
                "adata_sec", "df_gene_perturbation", "generated_realisation",
                "adata_reference_expression", "list_overall_loss",
                "list_trackinginfo_computeloss", "list_trackinginfo_projection",
                "dict_debug_info", "adata_insilico_gen_wihout_guidance",
                "adata_insilico_gen_wih_guidance", "adata_with_and_without_guidance",
            ]:
                if var_name in locals():
                    try:
                        del locals()[var_name]
                    except Exception:
                        pass

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    del adata_file
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

save_pickle(all_results, os.path.join(path_results, "all_results_index.pkl"))

print("\nDone.")
print(f"Processed sections: {list(all_results.keys())}")

Using device: cuda:0
Loading model checkpoint...
Loading data_mintflow...

Batch index map from training data:
  inflow_BatchID_0  BK18_Lesional Baseline__X0027131_Region 7
  inflow_BatchID_1  BK18_Non-lesional Baseline__X0027131_Region 6
  inflow_BatchID_10  BK22_Non-lesional Baseline__BK22-SKI-27-FO-2
  inflow_BatchID_11  BK23_Lesional Baseline__BK23-SKI-27-FO-1
  inflow_BatchID_12  BK23_Non-lesional Baseline__BK23-SKI-27-FO-2
  inflow_BatchID_13  BK23_Week 12__BK23-SKI-27-FO-5
  inflow_BatchID_14  BK24_Lesional Baseline__BK24-SKI-28-FO-1
  inflow_BatchID_15  BK24_Non-lesional Baseline__BK24-SKI-28-FO-2
  inflow_BatchID_16  BK24_Week 12__BK24-SKI-28-FO-4
  inflow_BatchID_17  BK25_Lesional Baseline__BK25-SKI-27-FO-1
  inflow_BatchID_18  BK25_Non-lesional Baseline__BK25-SKI-27-FO-2
  inflow_BatchID_19  BK25_Week 12__BK25-SKI-27-FO-5
  inflow_BatchID_2  BK18_Week 12__BK18-SKI-27-FO-4-S6
  inflow_BatchID_20  BK27_Lesional Baseline__X0027175_Region 2
  inflow_BatchID_21  BK27_Non-lesional

Scanning files:   0%|          | 0/64 [00:00<?, ?it/s]


Matched files:
 - BK18_Lesional Baseline__X0027131_Region 7.h5ad -> ['BK18_Lesional Baseline']
 - BK20_Lesional Baseline__X0027131_Region 8.h5ad -> ['BK20_Lesional Baseline']
 - BK21_Lesional Baseline__BK21-SKI-28-FO-1-S6.h5ad -> ['BK21_Lesional Baseline']
 - BK22_Lesional Baseline__BK22-SKI-27-FO-1.h5ad -> ['BK22_Lesional Baseline']
 - BK23_Lesional Baseline__BK23-SKI-27-FO-1.h5ad -> ['BK23_Lesional Baseline']
 - BK24_Lesional Baseline__BK24-SKI-28-FO-1.h5ad -> ['BK24_Lesional Baseline']
 - BK25_Lesional Baseline__BK25-SKI-27-FO-1.h5ad -> ['BK25_Lesional Baseline']
 - BK27_Lesional Baseline__X0027175_Region 2.h5ad -> ['BK27_Lesional Baseline']
 - BK30_Lesional Baseline__BK30-SKI-27-FO-2.h5ad -> ['BK30_Lesional Baseline']
 - BK39_Lesional Baseline__BK39-SKI-27-FO-1.h5ad -> ['BK39_Lesional Baseline']

Processing file: BK18_Lesional Baseline__X0027131_Region 7.h5ad
Matching sample_id values in this file: ['BK18_Lesional Baseline']

-------------------------------------------------------

Evaluating on tissue section: 17:   0%|          | 0/8 [00:00<?, ?it/s]

Generating the realisations of the expression data (i.e. generative samples) for the provided in silico tissue…

...

Double-checking the processing of `df_gene_perturbation`:   0%|          | 0/12949 [00:00<?, ?it/s]




Successfuly double-checked the processing of `df_gene_perturbation` !


Evaluating on tissue section: 17:   0%|          | 0/8 [00:00<?, ?it/s]

/nfs/team361/aa36/OnGit/Branches_MintFlow/feature-add-gene-perturbation/mintflow/src/mintflow/generativemodel.py:993: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(edge_index),


Guiding the embeddings:   0%|          | 0/1000 [00:00<?, ?it/s]